# Import Library

In [1]:
import requests
import time
import os
from dotenv import load_dotenv
import pandas as pd

# Setting & Checks Params

In [2]:
load_dotenv()

API_KEY = os.getenv("X_API_KEY")

if not API_KEY:
    raise ValueError("API Tidak ditemukan/error")
else:
    print('API key ada dan terakses')

API key ada dan terakses


Kasus OTT KPK Bupati Pati Sudewo mencuat pada 19 Januari 2026. Maka dari itu Query search untuk advanced searh harusnya min dari tanggal ini:

`since:2026-01-17_23:59:59_UTC` agar desah desuh tentang hal ini mungkin juga bisa ke-cover

Adapun keyword search yang akan digunakan sebagai berikut: 
* `("Sudewo" AND "KPK") OR ("Sudewo" AND "OTT") OR "Bupati Pati Sudewo"`

# Scrape tweets from advanced search

In [4]:
def scrape_twitter_advanced(query, max_tweets=5000, query_type='Latest'):

    # authentification
    try:
        load_dotenv()
        API_KEY = os.getenv("X_API_KEY")
        if not API_KEY: 
            raise ValueError("API key tidak ditemukan di .env")
    except Exception as e:
        print(f"Gagal API Key: {e}")
        return pd.DataFrame()

    url = "https://api.twitterapi.io/twitter/tweet/advanced_search"
    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }

    tweets_data = []
    seen_ids = set()
    cursor = ""
    retry_count = 0
    max_retries = 3

    print(f"Memulai scraping...")
    print(f"Query: {query}")
    print(f"Target: {max_tweets} tweets")

    while len(tweets_data) < max_tweets:
        try:
            querystring = {
                "query": query,
                "queryType": query_type,
                "cursor": cursor
            }

            response = requests.get(url, headers=headers, params=querystring)

            # Handling Limit (429)
            if response.status_code == 429:
                wait_time = (2 ** retry_count) * 5
                print(f"Kena Speed Limit (Error code: 429). Cooling down {wait_time} detik")
                time.sleep(wait_time)
                retry_count += 1
                if retry_count > max_retries:
                    print("Terlalu sering kena limit (Berhenti try new requests).")
                    break
                continue

            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text}")
                if response.status_code >= 500:
                    time.sleep(5)
                    continue
                else:
                    break
            
            retry_count = 0
            json_data = response.json()

            # check if there is tweeets in json_data
            if "tweets" not in json_data or not json_data["tweets"]:
                print("Tidak ada tweet lagi.")
                break

            new_tweets_count = 0
            for tweet in json_data["tweets"]:
                tweet_id = tweet.get("id")
                
                # skip if it's same tweet
                if tweet_id in seen_ids:
                    continue
                
                seen_ids.add(tweet_id)
                
                # get data
                tweet_info = {
                    "id": tweet_id,
                    "conversation_id": tweet.get("conversationId"),
                    "url": tweet.get("url"),
                    "text": tweet.get("text"),
                    "author": tweet.get("author", {}).get("userName"),
                    "author_location": tweet.get("author", {}).get("location"),
                    "reply_to": tweet.get("inReplyToUsername"),
                    "like_count": tweet.get("likeCount", 0),
                    "reply_count": tweet.get("replyCount", 0),
                    "retweet_count": tweet.get("retweetCount", 0),
                    "quote_count": tweet.get("quoteCount", 0),
                    "view_count": tweet.get("viewCount", 0),
                    "created_at": tweet.get("createdAt"),
                    "lang": tweet.get("lang"),
                    "hashtags": [h.get("text") for h in tweet.get("entities", {}).get("hashtags", [])]
                }
                tweets_data.append(tweet_info)
                new_tweets_count += 1
                
                if len(tweets_data) >= max_tweets: 
                    break

            print(f"   + Dapat {new_tweets_count} tweets baru (Total: {len(tweets_data)})")

            # Pagination Check
            next_cursor = json_data.get("next_cursor")
            has_next = json_data.get("has_next_page")

            if not has_next or not next_cursor or next_cursor == cursor:
                print("Pagination selesai.")
                break

            cursor = next_cursor

            # add sleep
            time.sleep(1.5) 

        except Exception as e:
            print(f"Error: {e}")
            break

    # convert to df
    df = pd.DataFrame(tweets_data)
    print(f"Selesai. Total {len(df)} tweets tersimpan.")
    
    return df

In [5]:
# Scraping Data "Top" (Populer)
print("Scraping Top Tweets")
main_query = '("Sudewo" AND "KPK") OR ("Sudewo" AND "OTT") OR "Bupati Pati Sudewo" since:2026-01-17'
df_top = scrape_twitter_advanced(
    main_query, 
    max_tweets=2000, 
    query_type='Top'
)

time.sleep(5)

# Scraping Data "Latest"
print("\nScraping Latest Tweets")
additional_query = '("Sudewo" AND "KPK") OR ("Sudewo" AND "OTT") OR "Bupati Pati Sudewo" since:2026-01-17'
df_latest = scrape_twitter_advanced(
    additional_query, 
    max_tweets=2000, 
    query_type='Latest'
)

#  concat data
print("\nMerging & Cleaning")
df_search_full = pd.concat([df_top, df_latest], ignore_index=True)

# drop duplicate by id
df_search_final = df_search_full.drop_duplicates(subset=['id'])

print(f"Total awal: {len(df_search_full)}")
print(f"Total bersih setelah drop duplicate: {len(df_search_final)}")

Scraping Top Tweets
Memulai scraping...
Query: ("Sudewo" AND "KPK") OR ("Sudewo" AND "OTT") OR "Bupati Pati Sudewo" since:2026-01-17
Target: 2000 tweets
   + Dapat 20 tweets baru (Total: 20)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 40)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 60)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 80)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 100)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 120)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 140)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 160)
Kena Speed Limit (Error code: 429). Cooling down 5 detik
   + Dapat 20 tweets baru (Total: 180)
Kena Speed Limit (Error code: 429). Cooling 

In [6]:
df_search_final.to_csv('tweets-data/tweets_from_search.csv', index=False)

# Scrape Quote and Reply from Viral Tweet

1. https://x.com/kompascom/status/2013214486900154533
2. https://x.com/kumparan/status/2013210519931654580
3. https://x.com/MurtadhaOne1/status/2013210033648234543
4. https://x.com/detikcom/status/2013474973415776705
5. https://x.com/Jateng_Twit/status/2013978983533305858
6. https://x.com/jhonsitorus_19/status/2013240346801561820
7. https://x.com/Urrangawak/status/2013766497294798952
8. https://x.com/Heraloebss/status/2013231292213842175

## Main tweet (viral)

In [8]:
def scrape_main_tweets(tweet_id):
    # Authentication moved inside the function
    try:
        load_dotenv()
        API_KEY = os.getenv("X_API_KEY")
        if not API_KEY: 
            raise ValueError("API key not found in .env")
    except Exception as e:
        print(f"Auth Error: {e}")
        return pd.DataFrame()

    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }

    url = "https://api.twitterapi.io/twitter/tweets"
    print(f"Scraping Main Tweet: {tweet_id}")
    
    retry_count = 0
    max_retries = 3

    while retry_count <= max_retries:
        try:
            # Parameter: tweet_ids
            querystring = {"tweet_ids": tweet_id}
            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 429:
                wait = 6 + (retry_count * 2)
                print(f"   Rate Limit (429). Sleeping {wait}s...")
                time.sleep(wait)
                retry_count += 1
                continue

            if response.status_code != 200:
                print(f"   Error {response.status_code}: {response.text}")
                return pd.DataFrame()

            json_data = response.json()
            tweets_list = json_data.get("tweets", [])
            
            if not tweets_list:
                print("   Data not found.")
                return pd.DataFrame()

            t = tweets_list[0]
            
            # Extract hashtags safely
            hashtags_list = []
            if "entities" in t and "hashtags" in t["entities"]:
                hashtags_list = [h.get("text") for h in t["entities"]["hashtags"]]

            data = {
                "id": t.get("id"),
                "conversation_id": t.get("conversationId"), 
                "source_tweet_id": t.get("id"),    
                "relation_type": "main_tweet", 
                "text": t.get("text"),
                "author": t.get("author", {}).get("userName"),
                "author_location": t.get("author", {}).get("location"),
                "created_at": t.get("createdAt"),
                "like_count": t.get("likeCount", 0),
                "reply_count": t.get("replyCount", 0),
                "retweet_count": t.get("retweetCount", 0),
                "quote_count": t.get("quoteCount", 0),
                "view_count": t.get("viewCount", 0),
                "url": t.get("url"),
                "lang": t.get("lang"),
                "reply_to": t.get("inReplyToUsername"),
                "hashtags": hashtags_list
            }
            return pd.DataFrame([data])

        except Exception as e:
            print(f"   Exception: {e}")
            return pd.DataFrame()
            
    return pd.DataFrame()

In [9]:
# Main Execution for Viral Tweets
viral_urls = [
    "https://x.com/kompascom/status/2013214486900154533",
    "https://x.com/kumparan/status/2013210519931654580",
    "https://x.com/MurtadhaOne1/status/2013210033648234543",
    "https://x.com/detikcom/status/2013474973415776705",
    "https://x.com/Jateng_Twit/status/2013978983533305858",
    "https://x.com/jhonsitorus_19/status/2013240346801561820",
    "https://x.com/Urrangawak/status/2013766497294798952",
    "https://x.com/Heraloebss/status/2013231292213842175"
]

# Extract IDs
viral_ids = [url.split("/")[-1].split("?")[0] for url in viral_urls]
main_tweets_data = []

print(f"Processing {len(viral_ids)} Viral Tweets...")

for vid in viral_ids:
    df = scrape_main_tweets(vid)
    if not df.empty:
        main_tweets_data.append(df)
    
    # Sleep to be safe
    time.sleep(2)

# Finalize
if main_tweets_data:
    df_main_viral = pd.concat(main_tweets_data, ignore_index=True)
    print(f"\nTotal Viral Tweets Collected: {len(df_main_viral)}")
else:
    print("\nNo main tweets collected.")

Processing 8 Viral Tweets...
Scraping Main Tweet: 2013214486900154533
Scraping Main Tweet: 2013210519931654580
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013210033648234543
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013474973415776705
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013978983533305858
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013240346801561820
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013766497294798952
   Rate Limit (429). Sleeping 6s...
Scraping Main Tweet: 2013231292213842175
   Rate Limit (429). Sleeping 6s...

Total Viral Tweets Collected: 8


In [10]:
df_main_viral

,id,conversation_id,source_tweet_id,relation_type,text,author,author_location,created_at,like_count,reply_count,retweet_count,quote_count,view_count,url,lang,reply_to,hashtags
0,2013214486900154533,2013214486900154533,2013214486900154533,main_tweet,Bupati Pati Sudewo Kena OTT KPK\n\nBaca di: ht...,kompascom,Jakarta,Mon Jan 19 11:38:32 +0000 2026,1337,145,367,90,553922,https://x.com/kompascom/status/201321448690015...,in,,[]
1,2013210519931654580,2013210519931654580,2013210519931654580,main_tweet,Komisi Pemberantasan Korupsi (KPK) melakukan o...,kumparan,,Mon Jan 19 11:22:46 +0000 2026,2421,247,649,149,182934,https://x.com/kumparan/status/2013210519931654580,in,,"[newsupdate, update, news, oneliner, kumparan]"
2,2013210033648234543,2013210033648234543,2013210033648234543,main_tweet,"BREAKING NEWS!\n\nKabarnya Bupati Pati, Sudewo...",MurtadhaOne1,,Mon Jan 19 11:20:50 +0000 2026,870,58,266,18,323571,https://x.com/MurtadhaOne1/status/201321003364...,in,,[]
3,2013474973415776705,2013474973415776705,2013474973415776705,main_tweet,"Bupati Pati Sudewo tiba di Gedung KPK, Selasa ...",detikcom,,Tue Jan 20 04:53:36 +0000 2026,76,10,23,2,6947,https://x.com/detikcom/status/2013474973415776705,in,,[]
4,2013978983533305858,2013978983533305858,2013978983533305858,main_tweet,Orang Pati pancen Uwu\n\nBupatine ditetapke Te...,Jateng_Twit,Indonesia,Wed Jan 21 14:16:22 +0000 2026,995,25,222,7,14065,https://x.com/Jateng_Twit/status/2013978983533...,in,,[]
5,2013240346801561820,2013240346801561820,2013240346801561820,main_tweet,Dua Kepala Daerah yang ditangkap KPK hari ini ...,jhonsitorus_19,,Mon Jan 19 13:21:17 +0000 2026,1997,246,655,31,89450,https://x.com/jhonsitorus_19/status/2013240346...,in,,[]
6,2013766497294798952,2013766497294798952,2013766497294798952,main_tweet,Akhirnya Bupati Pati Sudewo yang arogan tumban...,Urrangawak,🇮🇩 Pancasila,Wed Jan 21 00:12:01 +0000 2026,4239,124,855,30,84559,https://x.com/Urrangawak/status/20137664972947...,in,,[]
7,2013231292213842175,2013231292213842175,2013231292213842175,main_tweet,BREAKING— Gagal dimakzulkan karena menantang r...,Heraloebss,Indonesia,Mon Jan 19 12:45:18 +0000 2026,2984,134,834,34,176468,https://x.com/Heraloebss/status/20132312922138...,in,,[]


## Quotes and Replies

In [12]:
def scrape_replies_by_id(tweet_id, max_replies=200):
    # Authentication
    try:
        load_dotenv()
        API_KEY = os.getenv("X_API_KEY")
        if not API_KEY: 
            raise ValueError("API key not found in .env")
    except Exception as e:
        print(f"Auth Error: {e}")
        return pd.DataFrame()

    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }

    # Endpoint V1
    url = "https://api.twitterapi.io/twitter/tweet/replies"
    
    replies_data = []
    seen_ids = set()
    cursor = None 
    retry_count = 0
    max_retries = 5 

    print(f"Scraping Replies: {tweet_id}")

    while len(replies_data) < max_replies:
        try:
            querystring = {"tweetId": tweet_id}
            if cursor:
                querystring["cursor"] = cursor

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 429:
                wait_time = (2 ** retry_count) * 5 
                print(f"   Rate Limit (429). Sleeping {wait_time}s...")
                time.sleep(wait_time)
                retry_count += 1
                if retry_count > max_retries: break
                continue 

            if response.status_code != 200:
                print(f"   Error {response.status_code}: {response.text}")
                if response.status_code >= 500:
                    time.sleep(5)
                    continue
                else:
                    break
            
            retry_count = 0
            json_data = response.json()

            # Key 'tweets' confirmed from your JSON
            current_batch = json_data.get("tweets", [])
            
            if not current_batch:
                print("   No more replies.")
                break

            for t in current_batch:
                rid = t.get("id")
                if rid in seen_ids: continue
                seen_ids.add(rid)
                
                # Extract Hashtags
                hashtags_list = [h.get("text") for h in t.get("entities", {}).get("hashtags", [])]

                replies_data.append({
                    "id": rid,
                    "conversation_id": t.get("conversationId"),
                    "source_tweet_id": tweet_id,    
                    "relation_type": "reply",       
                    "text": t.get("text"),
                    "author": t.get("author", {}).get("userName"),
                    "author_location": t.get("author", {}).get("location"),
                    "created_at": t.get("createdAt"),
                    "like_count": t.get("likeCount", 0),
                    "reply_count": t.get("replyCount", 0),
                    "retweet_count": t.get("retweetCount", 0),
                    "quote_count": t.get("quoteCount", 0),
                    "view_count": t.get("viewCount", 0),
                    "url": t.get("url"),
                    "lang": t.get("lang"),
                    "reply_to": t.get("inReplyToUsername"),
                    "hashtags": hashtags_list
                })
                
                if len(replies_data) >= max_replies: break
            
            has_next = json_data.get("has_next_page")
            next_cursor = json_data.get("next_cursor")

            if not has_next or not next_cursor or next_cursor == cursor:
                break
            
            cursor = next_cursor
            time.sleep(1) 

        except Exception as e:
            print(f"   Exception: {e}")
            break

    print(f"   Done. Total replies: {len(replies_data)}")
    return pd.DataFrame(replies_data)

def scrape_quotes_by_id(tweet_id, max_items=200):
    # Authentication
    try:
        load_dotenv()
        API_KEY = os.getenv("X_API_KEY")
        if not API_KEY: 
            raise ValueError("API key not found in .env")
    except Exception as e:
        print(f"Auth Error: {e}")
        return pd.DataFrame()

    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }

    url = "https://api.twitterapi.io/twitter/tweet/quotes"
    
    quotes_data = []
    seen_ids = set()
    cursor = None
    retry_count = 0
    max_retries = 5 

    print(f"Scraping Quotes: {tweet_id}")

    while len(quotes_data) < max_items:
        try:
            querystring = {
                "tweetId": tweet_id,
                "includeReplies": "true"
            }
            if cursor:
                querystring["cursor"] = cursor

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 429:
                wait_time = (2 ** retry_count) * 5 
                print(f"   Rate Limit (429). Sleeping {wait_time}s...")
                time.sleep(wait_time)
                retry_count += 1
                if retry_count > max_retries: break
                continue 

            if response.status_code != 200:
                print(f"   Error {response.status_code}")
                break
            
            retry_count = 0
            json_data = response.json()

            # Quotes use key 'tweets' confirmed from your JSON
            current_batch = json_data.get("tweets", [])
            
            if not current_batch:
                print("   No more quotes.")
                break

            for t in current_batch:
                rid = t.get("id")
                if rid in seen_ids: continue
                seen_ids.add(rid)
                
                # Extract Hashtags
                hashtags_list = [h.get("text") for h in t.get("entities", {}).get("hashtags", [])]

                quotes_data.append({
                    "id": rid,
                    "conversation_id": t.get("conversationId"),
                    "source_tweet_id": tweet_id,    
                    "relation_type": "quote",       
                    "text": t.get("text"),
                    "author": t.get("author", {}).get("userName"),
                    "author_location": t.get("author", {}).get("location"),
                    "created_at": t.get("createdAt"),
                    "like_count": t.get("likeCount", 0),
                    "reply_count": t.get("replyCount", 0),
                    "retweet_count": t.get("retweetCount", 0),
                    "quote_count": t.get("quoteCount", 0),
                    "view_count": t.get("viewCount", 0),
                    "url": t.get("url"),
                    "lang": t.get("lang"),
                    "reply_to": t.get("inReplyToUsername"), # Often None for quotes, but kept for consistency
                    "hashtags": hashtags_list
                })
                
                if len(quotes_data) >= max_items: break
            
            has_next = json_data.get("has_next_page")
            next_cursor = json_data.get("next_cursor")

            if not has_next or not next_cursor or next_cursor == cursor:
                break
            
            cursor = next_cursor
            time.sleep(1) 

        except Exception as e:
            print(f"   Exception: {e}")
            break

    print(f"   Done. Total quotes: {len(quotes_data)}")
    return pd.DataFrame(quotes_data)

In [13]:
# define viral tweets
viral_urls = [
    "https://x.com/kompascom/status/2013214486900154533",
    "https://x.com/kumparan/status/2013210519931654580",
    "https://x.com/MurtadhaOne1/status/2013210033648234543",
    "https://x.com/detikcom/status/2013474973415776705",
    "https://x.com/Jateng_Twit/status/2013978983533305858",
    "https://x.com/jhonsitorus_19/status/2013240346801561820",
    "https://x.com/Urrangawak/status/2013766497294798952",
    "https://x.com/Heraloebss/status/2013231292213842175"
]

viral_ids = [url.split("/")[-1].split("?")[0] for url in viral_urls]
all_data_frames = []

print(f"Starting Scraper for {len(viral_ids)} Tweet IDs...")

for vid in viral_ids:
    print(f"\nProcessing ID: {vid}")
    
    # Replies (Code status: 200)
    df_rep = scrape_replies_by_id(vid, max_replies=500)
    if not df_rep.empty:
        all_data_frames.append(df_rep)
    
    time.sleep(2)

    # Quotes (Code status: 200)
    df_quo = scrape_quotes_by_id(vid, max_items=500)
    if not df_quo.empty:
        all_data_frames.append(df_quo)
        
    print("Cooling down 5s")
    time.sleep(5)

print("\nFinalizing Data")
df_final = pd.DataFrame()

if all_data_frames:
    df_final = pd.concat(all_data_frames, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=['id'])

if not df_final.empty:
    print(f"Total Rows: {len(df_final)}")
    print(f"Replies: {len(df_final[df_final['relation_type'] == 'reply'])}")
    print(f"Quotes: {len(df_final[df_final['relation_type'] == 'quote'])}")
else:
    print("No data collected.")

Starting Scraper for 8 Tweet IDs...

Processing ID: 2013214486900154533
Scraping Replies: 2013214486900154533
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   No more replies.
   Done. Total replies: 113
Scraping Quotes: 2013214486900154533
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   No more quotes.
   Done. Total quotes: 67
Cooling down 5s

Processing ID: 2013210519931654580
Scraping Replies: 2013210519931654580
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). Sleeping 5s...
   Rate Limit (429). S

# Concat all data

In [17]:
print(df_search_final.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1298 entries, 0 to 1649
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               1298 non-null   object
 1   conversation_id  1298 non-null   object
 2   url              1298 non-null   object
 3   text             1298 non-null   object
 4   author           1298 non-null   object
 5   author_location  1298 non-null   object
 6   reply_to         196 non-null    object
 7   like_count       1298 non-null   int64 
 8   reply_count      1298 non-null   int64 
 9   retweet_count    1298 non-null   int64 
 10  quote_count      1298 non-null   int64 
 11  view_count       1298 non-null   int64 
 12  created_at       1298 non-null   object
 13  lang             1298 non-null   object
 14  hashtags         1298 non-null   object
dtypes: int64(5), object(10)
memory usage: 162.2+ KB
None


In [18]:
print(df_main_viral.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               8 non-null      object
 1   conversation_id  8 non-null      object
 2   source_tweet_id  8 non-null      object
 3   relation_type    8 non-null      object
 4   text             8 non-null      object
 5   author           8 non-null      object
 6   author_location  8 non-null      object
 7   created_at       8 non-null      object
 8   like_count       8 non-null      int64 
 9   reply_count      8 non-null      int64 
 10  retweet_count    8 non-null      int64 
 11  quote_count      8 non-null      int64 
 12  view_count       8 non-null      int64 
 13  url              8 non-null      object
 14  lang             8 non-null      object
 15  reply_to         8 non-null      object
 16  hashtags         8 non-null      object
dtypes: int64(5), object(12)
memory usage: 1

In [16]:
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1109 entries, 0 to 1113
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               1109 non-null   object
 1   conversation_id  1109 non-null   object
 2   source_tweet_id  1109 non-null   object
 3   relation_type    1109 non-null   object
 4   text             1109 non-null   object
 5   author           1109 non-null   object
 6   author_location  1109 non-null   object
 7   created_at       1109 non-null   object
 8   like_count       1109 non-null   int64 
 9   reply_count      1109 non-null   int64 
 10  retweet_count    1109 non-null   int64 
 11  quote_count      1109 non-null   int64 
 12  view_count       1109 non-null   int64 
 13  url              1109 non-null   object
 14  lang             1109 non-null   object
 15  reply_to         862 non-null    object
 16  hashtags         1109 non-null   object
dtypes: int64(5), object(12)
memory usage: 

In [19]:
df_search_final['relation_type'] = 'search_result'

# combine data
df_combined = pd.concat([df_main_viral, df_final, df_search_final], ignore_index=True)

# drop duplicates if any
df_combined_final = df_combined.drop_duplicates(subset=['id'], keep='first')

# check output
print(f"Total Baris: {len(df_combined_final)}")
print("\nSebaran Tipe Data:")
print(df_combined_final['relation_type'].value_counts(dropna=False))
print("\nCek Missing Values di Kolom Kunci:")
print(df_combined_final[['id', 'text', 'author', 'relation_type']].isna().sum())


Total Baris: 2386

Sebaran Tipe Data:
relation_type
search_result    1269
reply             857
quote             252
main_tweet          8
Name: count, dtype: int64

Cek Missing Values di Kolom Kunci:
id               0
text             0
author           0
relation_type    0
dtype: int64


C:\Users\ramad\AppData\Local\Temp\ipykernel_85500\2963904304.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_search_final['relation_type'] = 'search_result'


In [20]:
df_combined_final.to_csv('tweets-data/data_gabungan_pati_final.csv', index=False)